# Etapa 3 — Modelagem

Entrada: `../data/processed/dataset.csv`, da Etapa 2.
Saída: um classificador treinado, avaliado e interpretado, e um ranking dos
melhores horários da janela.

Um detalhe que define esta etapa: **não existe label pronta.** O site
classifica as condições para pesca, não para surf — e pesca gosta de mar
calmo, quase o oposto. Definir o que é uma condição boa para surfar é a
primeira tarefa aqui, e é uma decisão sua.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

DIR_PROC = Path("..") / "data" / "processed"
ARQUIVO_DADOS = DIR_PROC / "dataset.csv"

In [ ]:
SEED = 42
CLASSES = ["ruim", "boa", "otima"]
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)

## 1. Carregar o dataset processado

Confira que veio o que você espera: número de linhas, tipos, nulos. Se a
Etapa 2 mudou, isso aparece aqui.

In [ ]:
dados = pd.read_csv(ARQUIVO_DADOS, parse_dates=["timestamp"])

assert dados["timestamp"].is_monotonic_increasing
assert dados["timestamp"].is_unique
assert dados.isna().sum().sum() == 0

resumo_entrada = pd.DataFrame({
    "linhas": [len(dados)],
    "colunas": [dados.shape[1]],
    "inicio": [dados["timestamp"].min()],
    "fim": [dados["timestamp"].max()],
    "nulos": [int(dados.isna().sum().sum())],
})
display(resumo_entrada)
display(dados.dtypes.rename("tipo").to_frame())
display(dados.head())

## 2. Definir a label

Escreva, em código, a regra que separa condições ruins, boas e ótimas (ou
outro conjunto de classes, se preferir).

- Quais variáveis entram na regra: altura da onda, vento, maré, direção?
- Quais limiares? De onde vem o número — leitura, conversa com surfista,
  distribuição dos próprios dados? Registre a origem, mesmo que seja um chute
  informado.
- Como as classes ficaram distribuídas? Se 95% das horas caírem numa classe,
  a métrica do modelo vai mentir para você.

Documente a regra em markdown antes de seguir. Tudo depois disso depende dela.

### Regra adotada

A regra é uma hipótese didática, não um boletim profissional. Considero **ótima**
uma hora com onda de pelo menos 1,2 m, vento de no máximo 22 km/h, maré entre
0,5 e 2,3 m e maré subindo. Considero **ruim** onda abaixo de 1,1 m ou vento
acima de 24 km/h; as demais horas são **boas**. Os limites de onda e vento
foram escolhidos a partir da distribuição desta janela e precisam ser validados
por surfistas locais.

In [ ]:
def classificar_condicao(tabela: pd.DataFrame) -> pd.Series:
    """Aplica a regra declarada acima a cada hora da previsão."""
    otima = (
        tabela["onda_m"].ge(1.2)
        & tabela["vento_kmh"].le(22)
        & tabela["mare_m"].between(0.5, 2.3)
        & tabela["tendencia_mare"].eq("subindo")
    )
    ruim = tabela["onda_m"].lt(1.1) | tabela["vento_kmh"].gt(24)
    return pd.Series(np.select([otima, ruim], ["otima", "ruim"], default="boa"), index=tabela.index)

dados["condicao_surf"] = pd.Categorical(
    classificar_condicao(dados), categories=CLASSES, ordered=True
)
distribuicao_classes = dados["condicao_surf"].value_counts().reindex(CLASSES).rename_axis("condicao").to_frame("horas")
distribuicao_classes["percentual"] = (100 * distribuicao_classes["horas"] / len(dados)).round(1)
display(distribuicao_classes)

fig, eixo = plt.subplots(figsize=(6, 4))
sns.countplot(data=dados, x="condicao_surf", order=CLASSES, hue="condicao_surf",
              legend=False, palette=["#d95f02", "#7570b3", "#1b9e77"], ax=eixo)
eixo.set(title="Distribuição das condições definidas pela regra", xlabel="condição", ylabel="horas")
plt.tight_layout()
plt.show()

## 3. Features e alvo

Separe `X` e `y`.

- Quais colunas entram como feature? Cuidado: usar diretamente as variáveis da
  sua regra faz o modelo apenas redescobrir o `if` que você escreveu. Isso é
  esperado aqui — mas você precisa saber que é isso que está acontecendo.
- Direção do vento e da onda são texto. Como transformar em número sem inventar
  uma ordem que não existe?

### Escolhas

Entram as variáveis físicas usadas na regra, o coeficiente e as direções.
`timestamp` fica de fora para não permitir que o modelo decore um dia ou uma
hora específica; `condicao_surf` é o alvo e não pode entrar em `X`. As direções
viram indicadores *one-hot*: isso não inventa uma ordem numérica entre E, ESE
e SE. Para o KNN, as variáveis numéricas são padronizadas porque ele mede
distâncias; a árvore recebe os valores nas unidades originais para facilitar a
interpretação de seus cortes.

In [ ]:
colunas_numericas = ["onda_m", "vento_kmh", "mare_m", "coeficiente"]
colunas_categoricas = ["direcao_onda_m", "direcao_vento_kmh", "tendencia_mare"]
colunas_features = colunas_numericas + colunas_categoricas

X = dados[colunas_features].copy()
y = dados["condicao_surf"].copy()
print(f"X: {X.shape} | y: {y.shape}")
display(X.head())

## 4. Treino e teste

Divida os dados. Vale estratificar pelas classes? A ordem temporal importa —
sortear linhas de um mesmo dia entre treino e teste vaza informação?

### Escolha

A separação respeita a ordem temporal: treino nos seis primeiros dias e teste
no último. Não estratifico nem sorteio linhas, pois horas vizinhas têm
condições muito parecidas e isso vazaria informação do futuro para o treino.
O teste representa, de forma mais honesta, usar dados passados para classificar
um dia posterior. Como há somente 24 horas no teste, a seção 6 complementa esse
resultado com validação cruzada temporal no período de treino.

In [ ]:
ultimo_dia = dados["timestamp"].dt.normalize().max()
mascara_treino = dados["timestamp"].dt.normalize().lt(ultimo_dia)
X_treino, X_teste = X.loc[mascara_treino], X.loc[~mascara_treino]
y_treino, y_teste = y.loc[mascara_treino], y.loc[~mascara_treino]

divisao = pd.concat({
    "treino": y_treino.value_counts().reindex(CLASSES, fill_value=0),
    "teste (último dia)": y_teste.value_counts().reindex(CLASSES, fill_value=0),
}, axis=1)
display(divisao)
print(f"Treino: {len(X_treino)} horas até {dados.loc[mascara_treino, 'timestamp'].max():%d/%m/%Y %H:%M}.")
print(f"Teste: {len(X_teste)} horas em {ultimo_dia:%d/%m/%Y}.")

## 5. Treinar

Treine pelo menos dois modelos de famílias diferentes (por exemplo uma árvore
e um modelo baseado em distância) para ter com o que comparar.

In [ ]:
preprocessador_arvore = ColumnTransformer(
    [("numericas", "passthrough", colunas_numericas),
     ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), colunas_categoricas)],
    verbose_feature_names_out=False,
)
preprocessador_knn = ColumnTransformer(
    [("numericas", StandardScaler(), colunas_numericas),
     ("categoricas", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas)]
)

arvore = Pipeline([("preprocessamento", preprocessador_arvore),
                  ("modelo", DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=SEED))])
knn = Pipeline([("preprocessamento", preprocessador_knn),
                ("modelo", KNeighborsClassifier(n_neighbors=7, weights="distance"))])
baseline = Pipeline([("preprocessamento", preprocessador_arvore),
                     ("modelo", DummyClassifier(strategy="most_frequent"))])
modelos = {"Árvore": arvore, "KNN": knn, "Baseline (classe mais comum)": baseline}

In [ ]:
for modelo in modelos.values():
    modelo.fit(X_treino, y_treino)

for nome, modelo in modelos.items():
    f1_treino = f1_score(y_treino, modelo.predict(X_treino), labels=CLASSES, average="macro", zero_division=0)
    print(f"{nome}: F1 macro no treino = {f1_treino:.2f}")

## 6. Avaliar

- Acurácia é suficiente aqui, com as classes desbalanceadas como estão?
- Olhe a matriz de confusão: que erro o modelo comete?
- Um único split é confiável com poucas linhas, ou vale validação cruzada?

In [ ]:
predicoes_teste = {nome: modelo.predict(X_teste) for nome, modelo in modelos.items()}
metricas = pd.DataFrame([
    {
        "modelo": nome,
        "acuracia_teste": accuracy_score(y_teste, previsao),
        "f1_macro_teste": f1_score(y_teste, previsao, labels=CLASSES, average="macro", zero_division=0),
    }
    for nome, previsao in predicoes_teste.items()
]).set_index("modelo").round(3)
display(metricas)

for nome in ["Árvore", "KNN"]:
    print(f"\nRelatório — {nome}")
    print(classification_report(y_teste, predicoes_teste[nome], labels=CLASSES, zero_division=0))

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
for eixo, nome in zip(eixos, ["Árvore", "KNN"]):
    ConfusionMatrixDisplay.from_predictions(y_teste, predicoes_teste[nome], labels=CLASSES, cmap="Blues", colorbar=False, ax=eixo)
    eixo.set_title(nome)
plt.suptitle("Matrizes de confusão — último dia como teste", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# A validação usa apenas o trecho de treino; o último dia continua reservado.
f1_macro = make_scorer(f1_score, labels=CLASSES, average="macro", zero_division=0)
cv_temporal = TimeSeriesSplit(n_splits=4)
resultados_cv = []
for nome, modelo in modelos.items():
    resultado = cross_validate(clone(modelo), X_treino, y_treino, cv=cv_temporal,
                              scoring={"acuracia": "accuracy", "f1_macro": f1_macro})
    resultados_cv.append({
        "modelo": nome,
        "acurácia temporal média": resultado["test_acuracia"].mean(),
        "F1 macro temporal médio": resultado["test_f1_macro"].mean(),
        "desvio do F1 macro": resultado["test_f1_macro"].std(),
    })
tabela_cv = pd.DataFrame(resultados_cv).set_index("modelo").round(3)
display(tabela_cv)
print("Leitura: F1 macro complementa a acurácia porque dá o mesmo peso às três classes.")
print("Com poucos dias, as variações entre cortes temporais são esperadas e devem ser tratadas como incerteza, não como precisão.")

## 7. Interpretar

Um modelo que acerta e você não sabe por quê não serve para responder nada.

- Plote a árvore. Os cortes que ela encontrou parecem com os seus limiares?
- Quais features o modelo considera mais importantes? Bate com sua intuição?
- Se a acurácia veio altíssima, explique o motivo — não celebre.

In [ ]:
nomes_transformados = arvore.named_steps["preprocessamento"].get_feature_names_out()
fig, eixo = plt.subplots(figsize=(18, 7))
plot_tree(
    arvore.named_steps["modelo"],
    feature_names=nomes_transformados,
    class_names=CLASSES,
    filled=True,
    rounded=True,
    impurity=False,
    ax=eixo,
)
eixo.set_title("Árvore de decisão aprendida no treino")
plt.tight_layout()
plt.show()

print(export_text(arvore.named_steps["modelo"], feature_names=list(nomes_transformados)))

In [ ]:
# Esta célula é autocontida: pode ser executada sem rodar antes o gráfico da árvore.
nomes_transformados = arvore.named_steps["preprocessamento"].get_feature_names_out()
importancias = pd.Series(arvore.named_steps["modelo"].feature_importances_, index=nomes_transformados)
importancias = importancias[importancias.gt(0)].sort_values()
fig, eixo = plt.subplots(figsize=(8, 4))
importancias.plot.barh(color="#177cb7", ax=eixo)
eixo.set(title="Importância das features na árvore", xlabel="redução de impureza", ylabel="feature")
plt.tight_layout()
plt.show()
display(importancias.sort_values(ascending=False).rename("importância").to_frame().round(3))

print("Interpretação: os cortes e as importâncias devem ser lidos como a reconstrução da regra usada para criar a label.")
print("Se a acurácia for alta, isso é esperado: onda, vento e maré entram tanto na regra quanto nas features.")

## 8. Responder a pergunta

Aplique o modelo na janela inteira e produza a resposta para Felipe e Nicholas:

- Quais os melhores dias da janela de previsão?
- Em média, qual o melhor horário do dia?
- Em que grau essa resposta é uma previsão do mar, e em que grau é apenas a
  sua regra reescrita?

In [ ]:
# Para recomendar a janela inteira, a árvore é treinada novamente com todos os dados disponíveis.
modelo_final = clone(arvore).fit(X, y)
probabilidades = pd.DataFrame(modelo_final.predict_proba(X), columns=modelo_final.classes_, index=dados.index)
for classe in CLASSES:
    if classe not in probabilidades:
        probabilidades[classe] = 0.0

ranking = dados[["timestamp", "onda_m", "vento_kmh", "mare_m", "tendencia_mare", "condicao_surf"]].copy()
ranking["previsao_modelo"] = modelo_final.predict(X)
# A nota de 0 a 2 ordena ruim, boa e ótima sem apresentar probabilidade física de surf.
ranking["nota_modelo"] = probabilidades["boa"] + 2 * probabilidades["otima"]
ranking = ranking.sort_values(["nota_modelo", "onda_m", "vento_kmh"], ascending=[False, False, True])

concordancia = (ranking["previsao_modelo"].astype(str) == ranking["condicao_surf"].astype(str)).mean()
resumo_dias = (
    ranking.assign(data=ranking["timestamp"].dt.date)
    .groupby("data", as_index=False)
    .agg(nota_media=("nota_modelo", "mean"), horas_previstas_otimas=("previsao_modelo", lambda s: (s == "otima").sum()))
    .sort_values(["nota_media", "horas_previstas_otimas"], ascending=False)
)
por_hora = (
    ranking.assign(hora=ranking["timestamp"].dt.hour)
    .groupby("hora", as_index=False)["nota_modelo"].mean()
    .sort_values("nota_modelo", ascending=False)
)

print(f"Concordância entre árvore e regra: {concordancia:.0%}.")
display(resumo_dias.assign(nota_media=lambda x: x["nota_media"].round(2)))
print("Dez horários mais bem ranqueados:")
display(ranking.head(10).assign(nota_modelo=lambda x: x["nota_modelo"].round(2)))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=resumo_dias.sort_values("data"), x="data", y="nota_media", color="#177cb7", ax=eixos[0])
eixos[0].set(title="Nota média do modelo por dia", xlabel="data", ylabel="nota (0 a 2)")
eixos[0].tick_params(axis="x", rotation=45)
sns.lineplot(data=por_hora.sort_values("hora"), x="hora", y="nota_modelo", marker="o", color="#1b9e77", ax=eixos[1])
eixos[1].set(title="Nota média do modelo por horário", xlabel="hora", ylabel="nota (0 a 2)", xticks=range(0, 24, 3))
plt.tight_layout()
plt.show()

melhor_dia = resumo_dias.iloc[0]
melhor_horario = ranking.iloc[0]
melhor_hora_media = int(por_hora.iloc[0]["hora"])
print(f"Melhor dia: {melhor_dia['data']:%d/%m}; melhor horário pontual: {melhor_horario['timestamp']:%d/%m às %H:%M}.")
print(f"Na média da janela, a hora mais bem ranqueada é {melhor_hora_media:02d}:00.")
print("Isto é principalmente a regra reescrita: o site prevê onda e vento; a árvore apenas traduz esses dados na nossa classificação.")

## 9. Conclusão e limites

Em texto:

1. A recomendação, em uma frase.
2. O teto deste modelo: ele reproduz a sua heurística, então é tão bom quanto
   ela. O que faria dela uma regra melhor?
3. O que a janela de ~7 dias e o volume de linhas impedem de afirmar.
4. Como você responderia "qual a melhor época do ano para surfar em João
   Pessoa" — que dado faltaria, e como coletá-lo.a

### Resposta

1. **Recomendação:** Felipe e Nicholas devem priorizar **14/10**, especialmente
   às **13:00**, quando a previsão reúne onda de 1,4 m, vento de 16 km/h e
   maré estimada de 0,61 m, segundo a regra adotada neste notebook.

2. **O teto do modelo é a regra.** A árvore aprende a reproduzir a heurística
   criada com onda, vento e maré; por isso, uma métrica alta não prova que ela
   prevê a qualidade do surf de forma independente. A regra melhoraria com
   limiares validados por surfistas locais, período e direção do swell, rajadas
   e direção do vento em relação ao pico, além de rótulos reais de sessões.

3. **Sete dias e 168 horas** não permitem concluir qual mês ou estação é melhor,
   generalizar o desempenho para outras semanas, estimar probabilidades anuais
   confiáveis ou estabelecer causalidade. O teste final tem somente 24 horas;
   portanto, sua métrica é instável e deve ser lida junto da validação temporal.

4. Para responder **qual é a melhor época do ano**, faltam vários anos de
   observações horárias de onda, período e direção do swell, vento, maré e
   condições locais, acompanhados de uma avaliação de qualidade feita por
   surfistas. A coleta pode ser automatizada diariamente e arquivada; então o
   modelo deve ser treinado em anos anteriores e testado em um período futuro
   inteiro.